# Compute Abnormal Returns

This notebook computes abnormal returns (AR) for each event-time observation.

The abnormal return formula is:

\[
AR_t = R_{stock,t} - (\hat{\alpha} + \hat{\beta} R_{market,t})
\]

where:
- \(R_{stock,t}\) is the stock return on day \(t\)
- \(R_{market,t}\) is the NASDAQ market return on day \(t\)
- \(\hat{\alpha}\) and \(\hat{\beta}\) are the event-specific market-model estimates

This notebook applies the estimated market model to the event-time panel created earlier.

It does **not** compute cumulative abnormal returns (CAR) yet. It only computes daily abnormal returns.

In [1]:
import pandas as pd
from pathlib import Path

## 1. Define file paths

In [2]:
event_windows_path = Path("../data/processed/event_windows.csv")
market_model_path = Path("../data/processed/event_market_model.csv")

output_path = Path("../data/processed/event_abnormal_returns.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

## 2. Load datasets

We load:
- the event-time return panel
- the event-specific market-model coefficients

In [3]:
event_windows = pd.read_csv(event_windows_path)
market_model = pd.read_csv(market_model_path)

print("event_windows shape:", event_windows.shape)
print("market_model shape:", market_model.shape)

event_windows shape: (24628, 12)
market_model shape: (188, 15)


## 3. Parse dates

In [4]:
event_windows["Date"] = pd.to_datetime(event_windows["Date"], errors="coerce")
event_windows["event_trading_day_final"] = pd.to_datetime(
    event_windows["event_trading_day_final"], errors="coerce"
)

market_model["event_trading_day_final"] = pd.to_datetime(
    market_model["event_trading_day_final"], errors="coerce"
)

display(event_windows.head())
display(market_model.head())

,event_id,ticker,file_name,call_datetime_et,after_market_close_et,event_trading_day_final,Date,t,Adj Close,return,index_adj_close,market_return
0,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-05,-120,25.767363,0.006629,5139.939941,0.006736
1,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-06,-119,25.823435,0.002176,5056.439941,-0.016245
2,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-07,-118,25.910910,0.003387,5043.540039,-0.002551
3,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-10,-117,26.852962,0.036357,5101.799805,0.011551
4,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-11,-116,25.455585,-0.052038,5036.790039,-0.012743


,event_id,ticker,event_trading_day_final,n_estimation_rows,alpha,beta,r_squared,adj_r_squared,alpha_pvalue,beta_pvalue,estimation_window,car_01_window,car_03_window,pre_vol_window,post_vol_window
0,1,AAPL,2016-01-27,101,-0.000387,1.102993,0.582332,0.578113,0.756382,1.772335e-20,True,True,True,True,True
1,2,AAPL,2016-04-27,101,-0.000282,1.079776,0.604705,0.600712,0.799526,1.140510e-21,True,True,True,True,True
2,3,AAPL,2016-07-27,101,-0.000466,0.854238,0.416435,0.410540,0.671931,3.227376e-13,True,True,True,True,True
3,4,AAPL,2016-10-26,101,0.001067,0.864596,0.339560,0.332889,0.335025,1.626726e-10,True,True,True,True,True
4,5,AAPL,2017-02-01,101,0.000399,0.885354,0.337867,0.331178,0.657605,1.850954e-10,True,True,True,True,True


## 4. Keep only model columns needed for abnormal return calculation

In [5]:
model_cols = [
    "event_id",
    "ticker",
    "event_trading_day_final",
    "alpha",
    "beta",
]

market_model_small = market_model[model_cols].copy()
display(market_model_small.head())

,event_id,ticker,event_trading_day_final,alpha,beta
0,1,AAPL,2016-01-27,-0.000387,1.102993
1,2,AAPL,2016-04-27,-0.000282,1.079776
2,3,AAPL,2016-07-27,-0.000466,0.854238
3,4,AAPL,2016-10-26,0.001067,0.864596
4,5,AAPL,2017-02-01,0.000399,0.885354


## 5. Merge market-model coefficients onto the event-time panel

Each event-time observation must inherit the correct event-specific \(\hat{\alpha}\) and \(\hat{\beta}\).

In [6]:
event_ar = event_windows.merge(
    market_model_small,
    on=["event_id", "ticker", "event_trading_day_final"],
    how="left",
    validate="many_to_one",
)

print("event_ar shape:", event_ar.shape)
display(event_ar.head())

event_ar shape: (24628, 14)


,event_id,ticker,file_name,call_datetime_et,after_market_close_et,event_trading_day_final,Date,t,Adj Close,return,index_adj_close,market_return,alpha,beta
0,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-05,-120,25.767363,0.006629,5139.939941,0.006736,-0.000387,1.102993
1,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-06,-119,25.823435,0.002176,5056.439941,-0.016245,-0.000387,1.102993
2,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-07,-118,25.910910,0.003387,5043.540039,-0.002551,-0.000387,1.102993
3,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-10,-117,26.852962,0.036357,5101.799805,0.011551,-0.000387,1.102993
4,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-11,-116,25.455585,-0.052038,5036.790039,-0.012743,-0.000387,1.102993


## 6. Check for missing coefficients after merge

All event-time rows should receive valid \(\hat{\alpha}\) and \(\hat{\beta}\) values.

In [7]:
print("Missing alpha after merge:", event_ar["alpha"].isna().sum())
print("Missing beta after merge:", event_ar["beta"].isna().sum())

assert event_ar["alpha"].notna().all(), "Some event-time rows are missing alpha estimates"
assert event_ar["beta"].notna().all(), "Some event-time rows are missing beta estimates"

Missing alpha after merge: 0
Missing beta after merge: 0


## 7. Compute expected returns

The expected return under the market model is:

\[
\hat{R}_{stock,t} = \hat{\alpha} + \hat{\beta} R_{market,t}
\]

In [8]:
event_ar["expected_return"] = (
    event_ar["alpha"] + event_ar["beta"] * event_ar["market_return"]
)

## 8. Compute abnormal returns

Abnormal return is the difference between actual stock return and expected return.

In [9]:
event_ar["abnormal_return"] = event_ar["return"] - event_ar["expected_return"]

## 9. Inspect results

In [10]:
display(
    event_ar[
        [
            "event_id",
            "ticker",
            "event_trading_day_final",
            "Date",
            "t",
            "return",
            "market_return",
            "alpha",
            "beta",
            "expected_return",
            "abnormal_return",
        ]
    ].head(20)
)

,event_id,ticker,event_trading_day_final,Date,t,return,market_return,alpha,beta,expected_return,abnormal_return
0,1,AAPL,2016-01-27,2015-08-05,-120,0.006629,0.006736,-0.000387,1.102993,0.007043,-0.000413
1,1,AAPL,2016-01-27,2015-08-06,-119,0.002176,-0.016245,-0.000387,1.102993,-0.018305,0.020481
2,1,AAPL,2016-01-27,2015-08-07,-118,0.003387,-0.002551,-0.000387,1.102993,-0.003201,0.006588
3,1,AAPL,2016-01-27,2015-08-10,-117,0.036357,0.011551,-0.000387,1.102993,0.012354,0.024003
4,1,AAPL,2016-01-27,2015-08-11,-116,-0.052038,-0.012743,-0.000387,1.102993,-0.014442,-0.037597
5,1,AAPL,2016-01-27,2015-08-12,-115,0.015420,0.001509,-0.000387,1.102993,0.001278,0.014142
6,1,AAPL,2016-01-27,2015-08-13,-114,-0.000781,-0.002147,-0.000387,1.102993,-0.002755,0.001974
7,1,AAPL,2016-01-27,2015-08-14,-113,0.007034,0.002916,-0.000387,1.102993,0.002830,0.004204
8,1,AAPL,2016-01-27,2015-08-17,-112,0.010348,0.008609,-0.000387,1.102993,0.009109,0.001239
9,1,AAPL,2016-01-27,2015-08-18,-111,-0.005633,-0.006353,-0.000387,1.102993,-0.007395,0.001761


## 10. Validate missing abnormal returns

There should be no missing abnormal returns if all required inputs are present.

In [11]:
print("Missing stock returns:", event_ar["return"].isna().sum())
print("Missing market returns:", event_ar["market_return"].isna().sum())
print("Missing expected returns:", event_ar["expected_return"].isna().sum())
print("Missing abnormal returns:", event_ar["abnormal_return"].isna().sum())

assert event_ar["return"].notna().all(), "Some stock returns are missing"
assert event_ar["market_return"].notna().all(), "Some market returns are missing"
assert event_ar["expected_return"].notna().all(), "Some expected returns are missing"
assert event_ar["abnormal_return"].notna().all(), "Some abnormal returns are missing"

Missing stock returns: 0
Missing market returns: 0
Missing expected returns: 0
Missing abnormal returns: 0


## 11. Inspect abnormal return distribution

This helps identify suspicious values before CAR is computed.

In [12]:
display(
    event_ar[["expected_return", "abnormal_return"]].describe()
)

,expected_return,abnormal_return
count,24628.000000,24628.000000
mean,0.001556,0.000003
std,0.016730,0.017694
min,-0.210599,-0.248430
25%,-0.004142,-0.006658
50%,0.001809,-0.000401
75%,0.008856,0.006148
max,0.160516,0.531674


## 12. Check extreme abnormal returns

Large abnormal returns can occur around earnings events, but extreme values should still be inspected.

In [13]:
extreme_ar = event_ar[event_ar["abnormal_return"].abs() > 0.5].copy()

print("Extreme abnormal returns:", len(extreme_ar))
display(
    extreme_ar[
        [
            "event_id",
            "ticker",
            "Date",
            "t",
            "return",
            "market_return",
            "expected_return",
            "abnormal_return",
        ]
    ].head(20)
)

Extreme abnormal returns: 2


,event_id,ticker,Date,t,return,market_return,expected_return,abnormal_return
2740,21,AMD,2016-04-22,0,0.522901,-0.008019,-0.008773,0.531674
2808,22,AMD,2016-04-22,-63,0.522901,-0.008019,0.000089,0.522812


The same calendar date can appear in multiple event windows.

Because:

t is RELATIVE to each event

## 13. Restrict to analysis window summary

The full event-time panel runs from `t = -120` to `t = +10`, but later CAR calculations will focus on:
- `[0,1]`
- `[0,3]`

This check confirms that abnormal returns exist throughout the event panel.

In [14]:
ar_counts = (
    event_ar.groupby("event_id")
    .size()
    .reset_index(name="n_rows")
)

display(ar_counts.head())
print("Minimum rows per event:", ar_counts["n_rows"].min())
print("Maximum rows per event:", ar_counts["n_rows"].max())

,event_id,n_rows
0,1,131
1,2,131
2,3,131
3,4,131
4,5,131


Minimum rows per event: 131
Maximum rows per event: 131


## 14. Finalize output columns

In [15]:
final_cols = [
    "event_id",
    "ticker",
    "file_name",
    "call_datetime_et",
    "after_market_close_et",
    "event_trading_day_final",
    "Date",
    "t",
    "Adj Close",
    "index_adj_close",
    "return",
    "market_return",
    "alpha",
    "beta",
    "expected_return",
    "abnormal_return",
]

final_cols = [c for c in final_cols if c in event_ar.columns]
event_ar = event_ar[final_cols].copy()

event_ar = event_ar.sort_values(["event_id", "t"]).reset_index(drop=True)
display(event_ar.head(20))

,event_id,ticker,file_name,call_datetime_et,after_market_close_et,event_trading_day_final,Date,t,Adj Close,index_adj_close,return,market_return,alpha,beta,expected_return,abnormal_return
0,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-05,-120,25.767363,5139.939941,0.006629,0.006736,-0.000387,1.102993,0.007043,-0.000413
1,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-06,-119,25.823435,5056.439941,0.002176,-0.016245,-0.000387,1.102993,-0.018305,0.020481
2,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-07,-118,25.910910,5043.540039,0.003387,-0.002551,-0.000387,1.102993,-0.003201,0.006588
3,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-10,-117,26.852962,5101.799805,0.036357,0.011551,-0.000387,1.102993,0.012354,0.024003
4,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-11,-116,25.455585,5036.790039,-0.052038,-0.012743,-0.000387,1.102993,-0.014442,-0.037597
5,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-12,-115,25.848108,5044.390137,0.015420,0.001509,-0.000387,1.102993,0.001278,0.014142
6,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-13,-114,25.827925,5033.560059,-0.000781,-0.002147,-0.000387,1.102993,-0.002755,0.001974
7,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-14,-113,26.009604,5048.240234,0.007034,0.002916,-0.000387,1.102993,0.002830,0.004204
8,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-17,-112,26.278761,5091.700195,0.010348,0.008609,-0.000387,1.102993,0.009109,0.001239
9,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-18,-111,26.130722,5059.350098,-0.005633,-0.006353,-0.000387,1.102993,-0.007395,0.001761


## 15. Save outputs

In [16]:
event_ar.to_csv(output_path, index=False)

print("Saved:")
print(output_path)

Saved:
../data/processed/event_abnormal_returns.csv


## 16. Validation summary

This summary reports:
- number of event-time rows
- number of events
- missing values in abnormal-return inputs
- count of extreme abnormal returns

In [17]:
summary = {
    "n_event_time_rows": len(event_ar),
    "n_events": int(event_ar["event_id"].nunique()),
    "missing_alpha": int(event_ar["alpha"].isna().sum()),
    "missing_beta": int(event_ar["beta"].isna().sum()),
    "missing_expected_return": int(event_ar["expected_return"].isna().sum()),
    "missing_abnormal_return": int(event_ar["abnormal_return"].isna().sum()),
    "extreme_abnormal_return_count": int(len(extreme_ar)),
}

summary_df = pd.DataFrame(summary.items(), columns=["check", "value"])
display(summary_df)

,check,value
0,n_event_time_rows,24628
1,n_events,188
2,missing_alpha,0
3,missing_beta,0
4,missing_expected_return,0
5,missing_abnormal_return,0
6,extreme_abnormal_return_count,2
